In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *

from makedf.mcstat import get_MCstat_unc

# turn off PerformanceWarning 
# triggered by mismatched column levels
from analysis_village.cc1pi.HelperFunctions import HelperFunctions
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/syst/"

today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-other-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
use_Ar23p = True

pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

if not use_Ar23p:
    mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV.df", keys2load, 100)
else:
    mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df", keys2load, 100)
    
mc_evt_df = mc_bnb_df['cc1pi']
mc_nu_df = mc_bnb_df['nudf']
mc_hdr_df = mc_bnb_df['hdr']

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']


pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()

#Perform the filtering to overload less the memmory
mc_evt_df = mc_evt_df[build_event_cumulative_masks(mc_evt_df, sideband = "")["energy"]]
cols_to_keep = [
        ('slc', 'self', '', '', '', ''),
        ('slc', 'tmatch', 'idx', '', '', ''),
        ('slc', 'nu_score', '', '', '', ''),
        ('slc', 'cut', 'obvious_cosmic', '', '', ''),
        ('slc', 'cut', 't0', '', '', ''),
        ('slc', 'cut', 'inside_FV', '', '', ''),
        ('slc', 'cut', 'nu_score', '', '', ''),
        ('slc', 'cut', 'track', '', '', ''),
        ('slc', 'cut', 'shower', '', '', ''),
        ('slc', 'cut', 'MIP_candidates', '', '', ''),
        ('slc', 'cut', 'angle', '', '', ''),
        ('slc', 'cut', 'proton_BDT', '', '', ''),
        ('slc', 'cut', 'proton_BDT_sideband', '', '', ''),
        ('slc', 'cut', 'containment', '', '', ''),
        ('slc', 'cut', 'michel', '', '', ''),
        ('slc', 'cut', 'extra_pion', '', '', ''),
        ('slc','cut','energy','','',''),
        ('slc', 'measure_var', 'angle_between_candidates', '', '', ''),
        ('slc', 'measure_var', 'num_protons', '', '', ''),
        ('slc','measure_var','reco_p_mu','','',''),
        ('slc','measure_var','reco_cos_theta_mu','','',''),
        ('slc','measure_var','TLE_p_pi','','',''),
        ('slc','measure_var','reco_cos_theta_pi','','',''), 
        ('slc','cut_var','n_MIP_candidates','','',''),
        ('slc','measure_var','delta_pT','','',''),
        ('slc','measure_var','delta_alpha_T','','',''),
        ('slc','measure_var','delta_phi_T','','','')
]
mc_evt_df = mc_evt_df[cols_to_keep]

mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()

# Perform selection

In [ ]:
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))

In [ ]:

#Do truth matchign
if not use_Ar23p:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_nu_df)
else:
    mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_nu_df) 
    print("Finished loading")    

In [ ]:
if not use_Ar23p:
    mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))
else:
    new_columns = []
    for c in mc_nu_df.columns:
        new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
    mc_nu_df.columns = pd.MultiIndex.from_tuples(new_columns)
    mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))

In [ ]:
HelperFunctions.print_purity(mc_evt_df, ('truth','nu_categ','','','',''))

# G4

In [ ]:
g4_systematics = [
    #'reinteractions_kminus_Geant4',
    #'reinteractions_kplus_Geant4',
    'reinteractions_neutron_Geant4',
    'reinteractions_piminus_Geant4',
    #'reinteractions_piplus_Geant4',
    'reinteractions_proton_Geant4'
]
label_map = {
    "reinteractions_kminus_Geant4": r"$K^{-}$",
    "reinteractions_kplus_Geant4": r"$K^{+}$",
    "reinteractions_neutron_Geant4": "n",
    "reinteractions_piminus_Geant4": r"$\pi^{-}$",  # Changed # to \
    "reinteractions_piplus_Geant4": r"$\pi^{+}$",   # Changed # to \
    "reinteractions_proton_Geant4": "p",
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_vs_energy = False   # <-- boolean switch
E_col = ('truth','E','','','','')
genie_categs = ["a"]#["other", "cosmic", "out_AV_nu", "nu_mu_NC" , "nu_mu_CC_QE" , "nu_mu_CC_MEC" , "nu_mu_CC_Res", "nu_mu_CC_Dis" ]
for genie_categ in genie_categs:
    #genie_df = mc_evt_df[mc_evt_df.truth.genie_categ == genie_categ].copy()
    
    for syst in g4_systematics:
        
        plt.figure()
    
        for i in range(101):
    
            col = ('truth', syst, f'univ_{i}', '', '', '')
    
            if col not in genie_df.columns:
                continue
    
            y = genie_df[col].values
            mask = np.isfinite(y)
    
            if plot_vs_energy:
    
                if E_col not in genie_df.columns:
                    raise ValueError("Energy column not found")
    
                x = genie_df[E_col].values
                mask &= np.isfinite(x)
    
                plt.scatter(x[mask], y[mask], s=3, alpha=0.3)
    
            else:
    
                x = np.full(mask.sum(), i)
                plt.scatter(x, y[mask], s=5)
    
        if plot_vs_energy:
            plt.xlabel("E")
        else:
            plt.xlabel("Universe")
    
        plt.ylabel("Value")
        plt.title(f"Syst: {syst}, genie categ = {genie_categ}")
    
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_vs_energy = True   # <-- boolean switch
E_col = ('truth','E','','','','')

for syst in g4_systematics:
    
    plt.figure()

    for i in range(101):

        col = ('truth', syst, f'univ_{i}', '', '', '')

        if col not in mc_evt_df.columns:
            continue

        y = mc_evt_df[col].values
        mask = np.isfinite(y)

        if plot_vs_energy:

            if E_col not in mc_evt_df.columns:
                raise ValueError("Energy column not found")

            x = mc_evt_df[E_col].values
            mask &= np.isfinite(x)

            plt.scatter(x[mask], y[mask], s=3, alpha=0.3)

        else:

            x = np.full(mask.sum(), i)
            plt.scatter(x, y[mask], s=5)

    if plot_vs_energy:
        plt.xlabel("E")
    else:
        plt.xlabel("Universe")

    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")

    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for syst in g4_systematics:

    means = []
    stds = []
    medians = []
    universes = []

    for i in range(101):

        col = ('truth', syst, f'univ_{i}', '', '', '')

        if col in mc_evt_df.columns:
            y = mc_evt_df[col].values
            y = y[~np.isnan(y)]
            
            universes.append(i)
            means.append(np.mean(y))
            stds.append(np.std(y))
            medians.append(np.median(y))

    means = np.array(means)
    stds = np.array(stds)
    medians = np.array(medians)
    plt.figure()

    # ±2σ band
    plt.bar(
        universes,
        4*stds,
        bottom=means-2*stds,
        width=0.8,
        alpha=0.2,
        label="±2σ"
    )

    # ±1σ band
    plt.bar(
        universes,
        2*stds,
        bottom=means-stds,
        width=0.5,
        alpha=0.5,
        label="±1σ"
    )

    # mean marker
    plt.scatter(universes, means, marker='o', s=40, label="Mean")

    # median marker
    plt.scatter(universes, medians, marker='x', s=40, label="Median")

    plt.xlabel("Universe")
    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")
    plt.legend()

    plt.show()

# GENIE

In [ ]:
genie_systematics_multisim = [
    'GENIEReWeight_SBN_v1_multisim_RPA_CCQE',
    'GENIEReWeight_SBN_v1_multisim_CoulombCCQE',
    'GENIEReWeight_SBN_v1_multisim_NormCCMEC',
    'GENIEReWeight_SBN_v1_multisim_NormNCMEC',
    'GENIEReWeight_SBN_v1_multisim_RDecBR1gamma',
    'GENIEReWeight_SBN_v1_multisim_RDecBR1eta',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC2pi',
    '#GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC2pi',
]

genie_systematics_multisigma = [
    "GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape",
    'GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE',
    "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC",
    "GENIEReWeight_SBN_v1_multisigma_Theta_Delta2Npi",
    "GENIEReWeight_SBN_v1_multisigma_ThetaDelta2NRad",
    "GENIEReWeight_SBN_v1_multisigma_MaCCRES",
    "GENIEReWeight_SBN_v1_multisigma_MaNCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvCCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvNCRES",
    'GENIEReWeight_SBN_v1_multisigma_AhtBY',
    'GENIEReWeight_SBN_v1_multisigma_BhtBY',
    'GENIEReWeight_SBN_v1_multisigma_CV1uBY',
    'GENIEReWeight_SBN_v1_multisigma_CV2uBY',
    "GENIEReWeight_SBN_v1_multisigma_NormCCCOH", # Handled by re-tuning
    "GENIEReWeight_SBN_v1_multisigma_NormNCCOH",
    'GENIEReWeight_SBN_v1_multisigma_MFP_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrCEx_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrInel_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrAbs_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrPiProd_pi',
    'GENIEReWeight_SBN_v1_multisigma_MFP_N',
    'GENIEReWeight_SBN_v1_multisigma_FrCEx_N',
    'GENIEReWeight_SBN_v1_multisigma_FrInel_N',
    'GENIEReWeight_SBN_v1_multisigma_FrAbs_N',
    'GENIEReWeight_SBN_v1_multisigma_FrPiProd_N',
    'GENIEReWeight_SBN_v1_multisigma_MaNCEL',
    'GENIEReWeight_SBN_v1_multisigma_EtaNCEL',
]



In [ ]:
print(mc_evt_df.truth.GENIEReWeight_SBN_v1_multisigma_FrCEx_N.columns)

In [ ]:
'''
for syst in genie_systematics_multisim:

    plt.figure()
    for i in range(101):  # universes 0..100
        col = ('truth', syst, f'univ_{i}', '', '', '')
        if col in mc_evt_df.columns:
            y = mc_evt_df[col].values
            x = [i] * len(y)  # x coordinate = universe index
            plt.scatter(x, y, s=5)

    plt.xlabel("Universe")
    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")
    plt.show()
'''

In [ ]:
'''
import matplotlib.pyplot as plt

shift_cols = ['cv','ps1','ps2','ps3','ms1','ms2','ms3']

for syst in genie_systematics_multisigma:
    print(syst)
    plt.figure()

    cols = [('truth', syst, shift, '', '', '') for shift in shift_cols]

    # keep only columns that exist
    cols = [c for c in cols if c in mc_evt_df.columns]

    df_syst = mc_evt_df[cols]

    # plot one line per row
    for _, row in df_syst.iterrows():
        plt.plot(range(len(cols)), row.values, alpha=0.2)

    plt.xticks(range(len(cols)), [c[2] for c in cols])
    plt.xlabel("Shift")
    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")

    plt.show()
'''

In [ ]:
for syst in genie_systematics_multisigma:
    print("Checking:", syst)

    morph_key = ('truth', syst, 'morph', '', '', '')
    ps_key    = ('truth', syst, 'ps1', '', '', '')

    if morph_key in mc_evt_df.columns:
        print("  Found morph")
        s_morph = mc_evt_df[morph_key]
        for i in range(100):
            seed_input = str(i) + str(syst)
            np.random.seed(hash(seed_input) % (2**32))
            wgt = (1 + (s_morph - 1) * 2 * np.abs(np.random.normal(0, 1))).clip(lower=0, upper=30)
            mc_evt_df[('truth', syst, f'univ_{i}', '', '', '')] = wgt

    elif ps_key in mc_evt_df.columns:
        print("  Found ps1")
        s_ps = mc_evt_df[ps_key]
        for i in range(100):
            seed_input = str(i) + str(syst)
            np.random.seed(hash(seed_input) % (2**32))
            wgt = (1 + (s_ps - 1) * np.random.normal(0, 1)).clip(lower=0, upper=30)
            mc_evt_df[('truth', syst, f'univ_{i}', '', '', '')] = wgt


In [ ]:
for syst in genie_systematics_multisigma:
    print("Checking:", syst)

    morph_key = ('truth', syst, 'morph', '', '', '')
    ps_key    = ('truth', syst, 'ps1', '', '', '')

    if morph_key in reduced_evt_df.columns:
        print("  Found morph")
        s_morph = reduced_evt_df[morph_key]
        for i in range(100):
            seed_input = str(i) + str(syst)
            np.random.seed(hash(seed_input) % (2**32))
            wgt = (1 + (s_morph - 1) * 2 * np.abs(np.random.normal(0, 1))).clip(lower=0, upper=30)
            reduced_evt_df[('truth', syst, f'univ_{i}', '', '', '')] = wgt

    elif ps_key in reduced_evt_df.columns:
        print("  Found ps1")
        s_ps = reduced_evt_df[ps_key]
        for i in range(100):
            seed_input = str(i) + str(syst)
            np.random.seed(hash(seed_input) % (2**32))
            wgt = (1 + (s_ps - 1) * np.random.normal(0, 1)).clip(lower=0, upper=30)
            reduced_evt_df[('truth', syst, f'univ_{i}', '', '', '')] = wgt


In [ ]:
print(mc_evt_df.truth.columns)

In [ ]:
genie_systematics_test = [
    'GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE',
    "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC",
]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_vs_energy = True   # <-- boolean switch
E_col = ('truth','E','','','','')

bins = np.linspace(0,4,100)
import matplotlib.pyplot as plt
bins = np.linspace(0, 4, 100)

# Example data (randomly distributed)


def plot_energy_comparison(df1, df2, energy_col=E_col, label1='Dataset 1', label2='Dataset 2'):
    """
    Plots energy data from two DataFrames on the same graph using Matplotlib.
    Assumes DataFrames have a DateTime index.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot first DataFrame
    ax.hist(df1[energy_col], bins=bins, edgecolor='blue', alpha=0.7,density = True)
    ax.hist(df2[energy_col], bins=bins, edgecolor='red', alpha=0.7,density = True)
    
    # Formatting
    ax.set_title('Energy Comparison', fontsize=14, fontweight='bold')
    ax.set_xlabel('E [GeV]', fontsize=12)
    ax.set_ylabel('A.U', fontsize=12) # Update unit as needed
    ax.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

# Execute the plot
plot_energy_comparison(mc_evt_df, reduced_evt_df)
plot_energy_comparison(mc_evt_df[mc_evt_df.truth.genie_categ == "nu_mu_CC_QE"], reduced_evt_df[reduced_evt_df.truth.genie_categ == "nu_mu_CC_QE"])
plot_energy_comparison(mc_evt_df[mc_evt_df.truth.genie_categ == "nu_mu_CC_MEC"], reduced_evt_df[reduced_evt_df.truth.genie_categ == "nu_mu_CC_MEC"])

dfs = [mc_evt_df, reduced_evt_df]
for syst in genie_systematics_test:

    for df in dfs:
        plt.figure()
    
        for i in range(101):
    
            col = ('truth', syst, f'univ_{i}', '', '', '')
    
            if col not in df.columns:
                continue
    
            y = df[col].values
            mask = np.isfinite(y)
    
            if plot_vs_energy:
    
                if E_col not in df.columns:
                    raise ValueError("Energy column not found")
                x = df[E_col].values
                mask &= np.isfinite(x)
                plt.scatter(x[mask], y[mask], s=3, alpha=0.3)
    
            else:
    
                x = np.full(mask.sum(), i)
                plt.scatter(x, y[mask], s=5)
    
        if plot_vs_energy:
            plt.xlabel("E")
        else:
            plt.xlabel("Universe")
    
        plt.ylabel("Value")
        plt.title(f"Syst: {syst}")
    
        plt.show()